# Stage 5-1: Optimizer 구현

이 노트북은 `src/core/optimizers.py`의 `SGD`와 `Adam` optimizer를 실습한다.
optimizer는 `model.params`와 `model.grads` 리스트를 참조하여 in-place로 파라미터를 업데이트한다.

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.models.mlp import MLP
from src.core.optimizers import SGD, Adam
from src.nn.losses import cross_entropy, cross_entropy_grad
from src.data.mnist import MnistDataset, get_task_spec
from src.data.dataloader import DataLoader

## 1. SGD

SGD(Stochastic Gradient Descent)는 가장 기본적인 optimizer이다.
현재 gradient 방향으로 learning rate만큼 파라미터를 이동한다.

$$\theta_t \leftarrow \theta_{t-1} - \eta \cdot g_t$$

`param -= lr * grad`는 NumPy 배열의 in-place 연산이다.
`param = param - lr * grad`와 달리 새 배열을 생성하지 않으므로
`model.params` 리스트가 참조하는 배열 객체가 유지된다.

In [ ]:
# SGD 생성 및 파라미터 참조 확인
model = MLP(task="multiclass", seed=42)
optimizer = SGD(model, lr=0.01)

print("model.params 수:", len(model.params))
print("optimizer.params 수:", len(optimizer.params))
print("동일 객체 참조:", all(p is q for p, q in zip(model.params, optimizer.params)))

In [ ]:
# SGD 한 스텝 — params가 in-place로 변경되는지 확인한다.
rng = np.random.default_rng(0)
x = rng.standard_normal((32, 784)).astype(np.float32)
y = rng.integers(0, 10, size=32).astype(np.int32)

params_before = [p.copy() for p in model.params]

logits = model.forward(x)
grad = cross_entropy_grad(logits, y)
model.backward(grad)
optimizer.step()

changed = [not np.allclose(p, b) for p, b in zip(model.params, params_before)]
print("step 후 변경된 파라미터:", sum(changed), "/", len(model.params))

In [ ]:
# 검증: lr=0 이면 파라미터가 변하지 않는다.
model0 = MLP(task="multiclass", seed=42)
opt0 = SGD(model0, lr=0.0)

params_before0 = [p.copy() for p in model0.params]
logits0 = model0.forward(x)
grad0 = cross_entropy_grad(logits0, y)
model0.backward(grad0)
opt0.step()

unchanged = [np.allclose(p, b) for p, b in zip(model0.params, params_before0)]
assert all(unchanged), "lr=0 이면 파라미터가 변하지 않아야 한다."
print("lr=0 검증 통과: 파라미터 불변")

## 2. Adam

Adam(Adaptive Moment Estimation)은 gradient의 1차 moment (이동 평균)와
2차 moment (제곱 이동 평균)를 유지하여 파라미터별 effective learning rate를 자동으로 조절한다.

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad (\text{1차 moment})$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad (\text{2차 moment})$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad (\text{bias correction})$$
$$\theta_t \leftarrow \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

bias correction은 $m_0 = 0$, $v_0 = 0$ 초기화로 인한 초기 편향을 제거한다.

In [ ]:
# Adam 초기화 확인 — m, v 배열이 params와 같은 shape으로 초기화된다.
model_a = MLP(task="multiclass", seed=42)
adam = Adam(model_a, lr=0.001)

print("Adam.ms shape 목록:", [m.shape for m in adam.ms])
print("Adam.vs shape 목록:", [v.shape for v in adam.vs])
print("초기 iter:", adam.iter)

In [ ]:
# Adam 한 스텝 — iter 증가, params 변경 확인
params_before_a = [p.copy() for p in model_a.params]

logits_a = model_a.forward(x)
grad_a = cross_entropy_grad(logits_a, y)
model_a.backward(grad_a)
adam.step()

print("step 후 iter:", adam.iter)
changed_a = [not np.allclose(p, b) for p, b in zip(model_a.params, params_before_a)]
print("변경된 파라미터:", sum(changed_a), "/", len(model_a.params))

assert adam.iter == 1
assert all(changed_a)
print("Adam 한 스텝 검증 통과")

In [ ]:
# m, v 배열 확인 — step 후 0이 아니어야 한다.
all_m_nonzero = all(not np.allclose(m, 0) for m in adam.ms)
all_v_nonzero = all(not np.allclose(v, 0) for v in adam.vs)
print("m 배열 업데이트 됨:", all_m_nonzero)
print("v 배열 업데이트 됨:", all_v_nonzero)

## 3. SGD vs Adam 수렴 비교

동일한 MLP에 SGD와 Adam을 각각 사용하여 20 스텝 학습하고 loss 곡선을 비교한다.

In [ ]:
def run_steps(model, optimizer, x, y, n_steps=20):
    losses = []
    for _ in range(n_steps):
        logits = model.forward(x)
        loss = cross_entropy(logits, y)
        grad = cross_entropy_grad(logits, y)
        model.backward(grad)
        optimizer.step()
        losses.append(float(loss))
    return losses

# 재현성을 위해 같은 seed
rng2 = np.random.default_rng(1)
x2 = rng2.standard_normal((64, 784)).astype(np.float32)
y2 = rng2.integers(0, 10, size=64).astype(np.int32)

m_sgd = MLP(task="multiclass", seed=42)
m_adam = MLP(task="multiclass", seed=42)

losses_sgd = run_steps(m_sgd, SGD(m_sgd, lr=0.01), x2, y2)
losses_adam = run_steps(m_adam, Adam(m_adam, lr=0.001), x2, y2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_sgd, label="SGD (lr=0.01)")
ax.plot(losses_adam, label="Adam (lr=0.001)")
ax.set_xlabel("Step")
ax.set_ylabel("Loss (cross-entropy)")
ax.set_title("SGD vs Adam: 20 Steps Loss")
ax.legend()
plt.tight_layout()
plt.show()

print(f"SGD 초기 loss: {losses_sgd[0]:.4f} -> 최종: {losses_sgd[-1]:.4f}")
print(f"Adam 초기 loss: {losses_adam[0]:.4f} -> 최종: {losses_adam[-1]:.4f}")

## 4. Learning Rate 영향 비교

SGD에서 learning rate 크기가 수렴에 미치는 영향을 시각화한다.

In [ ]:
lrs = [0.001, 0.01, 0.1]
fig, ax = plt.subplots(figsize=(9, 4))

for lr in lrs:
    m = MLP(task="multiclass", seed=42)
    opt = SGD(m, lr=lr)
    losses_lr = run_steps(m, opt, x2, y2, n_steps=30)
    ax.plot(losses_lr, label=f"lr={lr}")

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("SGD: Learning Rate 영향")
ax.legend()
plt.tight_layout()
plt.show()

## 5. DataLoader와 통합 — 3 task SGD 학습

실제 MNIST DataLoader를 사용하여 multiclass, binary, regression 3 task를 각각 5 epoch 학습한다.

In [ ]:
from src.nn.losses import binary_cross_entropy, binary_cross_entropy_grad, mse, mse_grad
from src.nn.metrics import accuracy, binary_accuracy, r2_score

DATASET_DIR = "/mnt/d/datasets/mnist"

_TASK_FNS = {
    "multiclass": (cross_entropy, cross_entropy_grad, accuracy),
    "binary": (binary_cross_entropy, binary_cross_entropy_grad, binary_accuracy),
    "regression": (mse, mse_grad, r2_score),
}

def train_epoch(model, loader, optimizer, loss_fn, grad_fn):
    total_loss, n_total = 0.0, 0
    for xb, yb in loader:
        logits = model.forward(xb)
        loss = loss_fn(logits, yb)
        grad = grad_fn(logits, yb)
        model.backward(grad)
        optimizer.step()
        total_loss += float(loss) * len(xb)
        n_total += len(xb)
    return total_loss / n_total

task_losses = {}
for task in ["multiclass", "binary", "regression"]:
    ds = MnistDataset("train", task, dataset_dir=DATASET_DIR)
    loader = DataLoader(ds, batch_size=128, shuffle=True)
    model_t = MLP(task=task, seed=42)
    opt_t = SGD(model_t, lr=0.01)
    loss_fn, grad_fn, _ = _TASK_FNS[task]
    epoch_losses = []
    for epoch in range(1, 6):
        ep_loss = train_epoch(model_t, loader, opt_t, loss_fn, grad_fn)
        epoch_losses.append(ep_loss)
        print(f"[{task}] Epoch {epoch}: loss={ep_loss:.4f}")
    task_losses[task] = epoch_losses
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for task, losses in task_losses.items():
    ax.plot(range(1, len(losses) + 1), losses, marker="o", label=task)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("3 Task SGD 학습 곡선 (5 epochs)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. SGD와 Adam 비교 정리

| 항목 | SGD | Adam |
|---|---|---|
| 업데이트 방향 | 현재 gradient $g_t$ | 1차 moment $\hat{m}_t$ (과거 gradient 포함) |
| 업데이트 크기 | 모든 파라미터 동일 $\eta$ | 파라미터별 $\eta / \sqrt{\hat{v}_t}$ |
| 추가 메모리 | 없음 | 파라미터당 $m$, $v$ 두 배열 |
| 하이퍼파라미터 | $\eta$ | $\eta$, $\beta_1$, $\beta_2$, $\epsilon$ |
| 수렴 안정성 | learning rate에 민감 | 넓은 learning rate 범위에서 안정적 |

**주요 설계 원칙**
- `param -= lr * grad` in-place 연산으로 `model.params` 참조 유지
- `Adam.ms`, `Adam.vs`는 `model.params`와 같은 shape, `iter` 추적으로 bias correction 적용
- `SGD`는 `Adam`보다 구현이 단순하며 적절한 learning rate에서 충분히 효과적이다